# Fine tuning gemma4

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
!pip install unsloth codecarbon
import unsloth
from unsloth import FastModel
from codecarbon import EmissionsTracker
import gc
import re
import os
from google.colab import drive
from PIL import Image
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

RECORES_DATASET_PATH = os.path.join(BASE_PATH, "dataset-recores/")


Mounted at /content/drive


In [ ]:
#@title Carga del dataset ReCoRES y creación de los conjuntos de entrenamiento, validación y test
import pandas as pd

train_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'train_modified.csv'), sep='\t')
val_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'dev_modified.csv'), sep='\t')
test_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'test_modified.csv'), sep='\t')

print("--- Información del Conjunto de Entrenamiento ---")
train_df.info()


--- Información del Conjunto de Entrenamiento ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      1047 non-null   object
 1   question  1047 non-null   object
 2   A         1047 non-null   object
 3   B         1047 non-null   object
 4   C         1047 non-null   object
 5   D         709 non-null    object
 6   E         380 non-null    object
 7   answer    1047 non-null   object
 8   reason    1047 non-null   object
dtypes: object(9)
memory usage: 73.7+ KB


In [ ]:

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    bias = "none",
    random_state = 3407,
)

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

OPTIONS = ["A", "B", "C", "D", "E"]

SYSTEM_PROMPT_TEXT = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer atentamente el texto y responder ÚNICAMENTE con la letra de la opción correcta.
Las ÚNICAS opciones de respuesta válidas son: {letras_validas}

Responde a la pregunta basándote EXCLUSIVAMENTE en la información del texto, sin utilizar conocimiento externo.

No escribas explicaciones, ni introducciones, ni repitas la pregunta.
SOLO la letra de la opción elegida."""

def get_valid_options(row):
    """Devuelve lista de letras con opción no nula en el orden original."""
    return [opt for opt in OPTIONS if pd.notna(row[opt]) and str(row[opt]).strip() != ""]

def build_conversation(row):
    valid_opts = get_valid_options(row)
    letras_validas = ", ".join(valid_opts)

    system_prompt = SYSTEM_PROMPT_TEXT.format(letras_validas=letras_validas)

    opciones_str = "\n".join(f"{opt}) {row[opt]}" for opt in valid_opts)
    user_content = f"Texto: {row['text']}\nPregunta: {row['question']}\nOpciones:\n{opciones_str}"

    assistant_content = str(row['answer']).strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts }

def prepare_dataset(df_input):
    """Función para automatizar la limpieza y formateo de cualquier split."""
    df_temp = df_input.copy()
    df_temp["conversations"] = df_temp.apply(build_conversation, axis=1)
    hf_ds = Dataset.from_pandas(df_temp[["conversations"]])
    return hf_ds.map(formatting_prompts_func, batched=True)

train_dataset = prepare_dataset(train_df)
val_dataset   = prepare_dataset(val_df)
test_dataset  = prepare_dataset(test_df)


Map:   0%|          | 0/1047 [00:00<?, ? examples/s]

Map:   0%|          | 0/363 [00:00<?, ? examples/s]

Map:   0%|          | 0/386 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset[-1]["text"])
print(train_dataset[4]["text"])

<|turn>system
Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer atentamente el texto y responder ÚNICAMENTE con la letra de la opción correcta.
Las ÚNICAS opciones de respuesta válidas son: A, B, C

Responde a la pregunta basándote EXCLUSIVAMENTE en la información del texto, sin utilizar conocimiento externo.

No escribas explicaciones, ni introducciones, ni repitas la pregunta.
SOLO la letra de la opción elegida.<turn|>
<|turn>user
Texto: Los sectores conservadores aman reprimir. Es la forma de conservar su poder. Reprimir y embrutecer. Mucha bala poca escuela. De ahí todo este rollo sobre la inseguridad. Millones salen y vuelven cada día a sus hogares sin ver, incluso, otra violencia que no sea la que genera la injusticia. Es decir, aquello que los medios no llaman violencia: niños mendigando, colas interminables para una cita médica, maestros peor pagados… imposible. Tuberculosis, mucha tuberculosis, con una "clase media" que accedi

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",

        per_device_train_batch_size = 16,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 2,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,

        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_steps = 12,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_gemma4_final",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

LORA_PATH = os.path.join(BASE_PATH, "ft_gemma4")

tracker = EmissionsTracker(
    project_name="ft_gemma4_final",
    output_dir=LORA_PATH,
    output_file="ft_gemma4_emissions.csv",
    log_level="warning"
)

tracker.start()
try:
    trainer_stats = trainer.train()
finally:
    emissions = tracker.stop()
    print(f"Entrenamiento completado.")
    print(f"Emisiones estimadas: {emissions:.4f} kg de CO2eq")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

...

[codecarbon WARNING @ 11:38:56] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 11:38:56] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 11:38:56] No CPU tracking mode found. Falling back on CPU constant mode.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,047 | Num Epochs = 3 | Total steps = 99
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-__

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.272207,0.935194
2,0.258408,0.928575
3,0.242678,0.941533


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs_gemma4_final/checkpoint-33/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_gemma4_final/checkpoint-66/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_gemma4_final/checkpoint-99/tokenizer_config.json.


Entrenamiento completado.
Emisiones estimadas: 0.0263 kg de CO2eq


In [ ]:
lora_path = os.path.join(BASE_PATH, "ft_gemma4")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"Adaptadores guardados en: {lora_path}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/ft_gemma4/tokenizer_config.json.


Adaptadores guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/ft_gemma4
